Перевiрити значущiсть коефiцiєнтiв моделi А i розрахувати їх iнтервали довiри.
Розрахувати точковий прогноз регресанда та iнтервальнi прогнози для математичного очiкування i для iндивiдуального значення регресанда для моделi А.
У якостi прогнозної точки взяти вектор середнiх значень регресорiв.

In [1]:
import pandas as pd
import numpy as np
from scipy import stats

data = {
    'Країна': [
        'Австралія', 'Австрія', 'Аргентина', 'Бангладеш', 'Бельгія', 'Білорусь', 'Бразилія', 'Буркіна-Фасо', 
        'Велика Британія', "В'єтнам", 'Гаїті', 'Гондурас', 'Гонконг', 'Ефіопія', 'Єгипет', 'Замбія', 'Індія', 
        'Ірландія', 'Іспанія', 'Італія', 'Канада', 'Китай', 'Колумбія', 'Коста-Рика', 'Куба', 'Малайзія', 
        'Марокко', 'Мексика', 'Нідерланди', 'Німеччина', 'Нова Зеландія', 'Норвегія', 'ОАЕ', 'ПАР', 
        'Південна Корея', 'Північна Корея', 'Польща', 'Португалія', 'Росія', 'Саудівська Аравія', 'Сінгапур', 
        'США', 'Таїланд', 'Туреччина', 'Україна', 'Філіппіни', 'Фінляндія', 'Франція', 'Чилі', 'Швейцарія', 
        'Швеція', 'Японія'
    ],
    'Y1': [74, 73, 68, 53, 73, 66, 57, 47, 74, 63, 43, 65, 75, 51, 60, 44, 58, 73, 74, 74, 74, 67, 69, 76, 74, 66, 66, 69, 75, 73, 73, 74, 70, 62, 68, 67, 69, 71, 64, 66, 73, 73, 65, 69, 65, 63, 72, 74, 71, 75, 75, 76],
    'X1': [17800, 8000, 33900, 125000, 10100, 10300, 156600, 10000, 58400, 73100, 6500, 5600, 5800, 55200, 60000, 9100, 911600, 3600, 39200, 58100, 29100, 1205200, 35600, 3300, 11100, 19500, 28600, 91800, 15400, 81200, 3524, 4300, 2800, 43900, 45000, 23100, 38600, 10500, 149200, 18000, 2900, 260800, 59400, 62200, 51800, 69800, 5100, 58000, 14000, 7000, 8800, 125500],
    'X2': [15, 12, 20, 35, 12, 13, 21, 47, 13, 27, 40, 35, 13, 45, 29, 46, 29, 14, 11, 11, 14, 21, 24, 26, 17, 29, 29, 28, 13, 11, 16, 13, 28, 34, 16, 24, 14, 12, 13, 38, 16, 15, 19, 26, 12, 27, 13, 13, 23, 12, 14, 11],
    'X3': [8, 11, 9, 11, 11, 11, 9, 18, 11, 8, 19, 6, 6, 14, 9, 18, 10, 9, 9, 10, 8, 7, 6, 4, 7, 5, 6, 5, 9, 11, 8, 10, 3, 8, 6, 6, 10, 10, 11, 6, 6, 9, 6, 6, 13, 7, 10, 9, 6, 9, 11, 7],
    'X4': [16848, 18396, 3408, 202, 17912, 6500, 2354, 357, 15974, 230, 383, 1030, 14641, 122, 748, 573, 275, 12170, 13047, 17500, 19904, 377, 1538, 2031, 1382, 2995, 1062, 3604, 17245, 17539, 14381, 17755, 14193, 3128, 6627, 1000, 4429, 9000, 6680, 6651, 14990, 23474, 1800, 3721, 2340, 867, 15877, 18944, 2591, 22384, 16900, 19860],
    'X5': [85, 58, 86, 16, 96, 65, 75, 15, 89, 20, 29, 44, 94, 12, 44, 42, 26, 57, 78, 69, 77, 26, 70, 47, 74, 43, 46, 73, 89, 85, 84, 75, 81, 49, 72, 60, 62, 34, 74, 77, 100, 75, 22, 61, 67, 43, 60, 73, 85, 62, 84, 77]
}
df = pd.DataFrame(data)

In [2]:
coefficients = {
    'X0': 8.21702248e+01,
    'X1': -1.95235588e-06,
    'X2': -3.74770557e-01,
    'X3': -1.15007499e+00,
    'X4': 3.26352360e-04,
    'X5': 1.51121836e-02
}

In [3]:
# діагональ з дисперсійно-коваріаційної матриці оцінених коефіцієнтів (lab3)
standard_errors = {
    'X0': 7.234633**0.5,
    'X1': 4.000000e-12**0.5,
    'X2': 3.204668e-03**0.5,
    'X3': 1.548347e-02**0.5,
    'X4': 5.664000e-09**0.5,
    'X5': 5.915398e-04**0.5
}

T = 52
N = len(coefficients)  # Кількість параметрів (6)
alpha = 0.05 # Рівень значущості

# 1. Загальний вигляд моделі
model_terms = [f"{coefficients['X0']:.4e}"]
for i in range(1, N):
    coef = coefficients[f'X{i}']
    sign = "+" if coef >= 0 else "-"
    model_terms.append(f"{sign} {abs(coef):.4e}*X{i}")
    
print("=== ЗАГАЛЬНИЙ ВИГЛЯД МОДЕЛІ ===")
print(f"Y = {' '.join(model_terms)}")
print("\n")

# 2. Розрахунок t критичного (Крок 4 алгоритму)
t_kr = stats.t.ppf(1 - alpha/2, T - N) # таблиця Ст'юдента
print("=== КРИТИЧНЕ ЗНАЧЕННЯ ===")
print(f"t_кр = {t_kr:.4f} (при alpha={alpha}, df={T - N})\n")

# 3. Розрахунок t статистичного (Крок 3) та висновки (Кроки 1, 5)
print("=== ПЕРЕВІРКА ЗНАЧУЩОСТІ КОЕФІЦІЄНТІВ ===")
for key, beta in coefficients.items():
    sigma = standard_errors[key]
    
    # Крок 3 алгоритму: розрахунок t_ст
    t_st = beta / sigma
    
    # Крок 5 алгоритму: порівняння |t_ст| > t_кр
    if abs(t_st) > t_kr:
        conclusion = f"Відхиляємо H0. Коефіцієнт значимо відрізняється від нуля (приймається HA з імовірністю {(1-alpha)*100}%)."
    else:
        conclusion = "Приймаємо H0. Коефіцієнт не є статистично значущим."
        
    print(f"[{key}]:")
    print(f"  beta = {beta:e}, sigma = {sigma:e}")
    print(f"  t_ст = {t_st:.4f}")
    print(f"  |{t_st:.4f}| > {t_kr:.4f} ? {'Так' if abs(t_st) > t_kr else 'Ні'}")
    print(f"  Висновок: {conclusion}\n")

=== ЗАГАЛЬНИЙ ВИГЛЯД МОДЕЛІ ===
Y = 8.2170e+01 - 1.9524e-06*X1 - 3.7477e-01*X2 - 1.1501e+00*X3 + 3.2635e-04*X4 + 1.5112e-02*X5


=== КРИТИЧНЕ ЗНАЧЕННЯ ===
t_кр = 2.0129 (при alpha=0.05, df=46)

=== ПЕРЕВІРКА ЗНАЧУЩОСТІ КОЕФІЦІЄНТІВ ===
[X0]:
  beta = 8.217022e+01, sigma = 2.689727e+00
  t_ст = 30.5496
  |30.5496| > 2.0129 ? Так
  Висновок: Відхиляємо H0. Коефіцієнт значимо відрізняється від нуля (приймається HA з імовірністю 95.0%).

[X1]:
  beta = -1.952356e-06, sigma = 2.000000e-06
  t_ст = -0.9762
  |-0.9762| > 2.0129 ? Ні
  Висновок: Приймаємо H0. Коефіцієнт не є статистично значущим.

[X2]:
  beta = -3.747706e-01, sigma = 5.660979e-02
  t_ст = -6.6202
  |-6.6202| > 2.0129 ? Так
  Висновок: Відхиляємо H0. Коефіцієнт значимо відрізняється від нуля (приймається HA з імовірністю 95.0%).

[X3]:
  beta = -1.150075e+00, sigma = 1.244326e-01
  t_ст = -9.2426
  |-9.2426| > 2.0129 ? Так
  Висновок: Відхиляємо H0. Коефіцієнт значимо відрізняється від нуля (приймається HA з імовірністю 95.0%)

In [4]:
print(f"Розрахунок інтервалів довіри (рівень довіри {(1-alpha)*100:.0f}%):")
print("-" * 60)

for key, beta_k in coefficients.items():
    sigma_k = standard_errors[key]
    
    # Розрахунок нижньої (НМ) та верхньої (ВМ) меж
    margin_of_error = t_kr * sigma_k
    lower_bound = beta_k - margin_of_error
    upper_bound = beta_k + margin_of_error
    
    # Перевірка входження нуля в інтервал
    contains_zero = lower_bound <= 0 <= upper_bound
    
    if contains_zero:
        conclusion = f"Нуль НАЛЕЖИТЬ довірчому інтервалу. Коефіцієнт незначимо відрізняється від нуля з імовірністю {(1-alpha)*100:.0f}%."
    else:
        conclusion = f"Нуль НЕ НАЛЕЖИТЬ довірчому інтервалу. Коефіцієнт значимо відрізняється від нуля з імовірністю {(1-alpha)*100:.0f}%."
        
    print(f"Коефіцієнт {key}: {beta_k:e}")
    print(f"Інтервал: [{lower_bound:e}; {upper_bound:e}]")
    print(f"Висновок: {conclusion}")
    print("-" * 60)

Розрахунок інтервалів довіри (рівень довіри 95%):
------------------------------------------------------------
Коефіцієнт X0: 8.217022e+01
Інтервал: [7.675608e+01; 8.758437e+01]
Висновок: Нуль НЕ НАЛЕЖИТЬ довірчому інтервалу. Коефіцієнт значимо відрізняється від нуля з імовірністю 95%.
------------------------------------------------------------
Коефіцієнт X1: -1.952356e-06
Інтервал: [-5.978147e-06; 2.073435e-06]
Висновок: Нуль НАЛЕЖИТЬ довірчому інтервалу. Коефіцієнт незначимо відрізняється від нуля з імовірністю 95%.
------------------------------------------------------------
Коефіцієнт X2: -3.747706e-01
Інтервал: [-4.887201e-01; -2.608210e-01]
Висновок: Нуль НЕ НАЛЕЖИТЬ довірчому інтервалу. Коефіцієнт значимо відрізняється від нуля з імовірністю 95%.
------------------------------------------------------------
Коефіцієнт X3: -1.150075e+00
Інтервал: [-1.400545e+00; -8.996052e-01]
Висновок: Нуль НЕ НАЛЕЖИТЬ довірчому інтервалу. Коефіцієнт значимо відрізняється від нуля з імовірністю 

In [ ]:
# lab3
# X_matrix - матриця регресорів (розміру T x N, де перший стовпець зазвичай одиниці)
# Y_vector - вектор фактичних значень регресанда (розміру T)
# beta_hat - вектор оцінених коефіцієнтів (розміру N)
# cov_matrix_beta - дисперсійно-коваріаційна матриця (розміру N x N)

x_cols = ['X1', 'X2', 'X3', 'X4', 'X5']
T = len(df)
N = len(x_cols) + 1
X_matrix = np.column_stack((np.ones(T), df[x_cols].values))
Y_vector = df['Y1'].values
XTX = X_matrix.T @ X_matrix
XTX_inv = np.linalg.inv(XTX)
beta_hat = XTX_inv @ X_matrix.T @ Y_vector
Y_pred = X_matrix @ beta_hat
residuals = Y_vector - Y_pred
sigma2_u = np.sum(residuals**2) / (T - N)
cov_matrix_beta = sigma2_u * XTX_inv

In [11]:
# Визначаємо розмірність вибірки (T) та кількість параметрів (N)
T, N = X_matrix.shape

# 1. Формування прогнозної точки x_tilde (вектор середніх значень регресорів)
x_tilde = np.mean(X_matrix, axis=0)

# 2. Розрахунок точкового прогнозу
y_pred = beta_hat.T @ x_tilde

# 3. Розрахунок стандартних похибок для інтервалів
# Похибка для математичного очікування (sigma_e)
sigma_e = np.sqrt(x_tilde.T @ cov_matrix_beta @ x_tilde)

# Похибка для індивідуального значення (sigma_e_i)
sigma_e_i = np.sqrt(sigma2_u + x_tilde.T @ cov_matrix_beta @ x_tilde)

# 4. Розрахунок прогнозних інтервалів (HM - нижня межа, BM - верхня межа)
# Для математичного очікування
me_HM = y_pred - t_kr * sigma_e
me_BM = y_pred + t_kr * sigma_e

# Для індивідуального значення
ind_HM = y_pred - t_kr * sigma_e_i
ind_BM = y_pred + t_kr * sigma_e_i

print("Вектор середніх значень регресорів (прогнозна точка x_tilde):")
print(x_tilde)

print(f"\nТочкове оцінене значення (прогноз): {y_pred:.4f}")

print("\nЗгідно з теорією, точкове оцінене значення має такі дві інтерпретації:")
print("- оцінка математичного очікування регресанда;")
print("- оцінка індивідуального значення регресанда.")

print("\nПрогнозний інтервал величини математичного очікування регресанда:")
print(f"HM : {me_HM:.4f}")
print(f"BM : {me_BM:.4f}")

print("\nІнтервал прогнозу для індивідуального значення регресанда:")
print(f"HM : {ind_HM:.4f}")
print(f"BM : {ind_BM:.4f}")

Вектор середніх значень регресорів (прогнозна точка x_tilde):
[1.00000000e+00 8.15946923e+04 2.13269231e+01 8.88461538e+00
 8.42190385e+03 6.16730769e+01]

Точкове оцінене значення (прогноз): 67.4808

Згідно з теорією, точкове оцінене значення має такі дві інтерпретації:
- оцінка математичного очікування регресанда;
- оцінка індивідуального значення регресанда.

Прогнозний інтервал величини математичного очікування регресанда:
HM : 66.7354
BM : 68.2261

Інтервал прогнозу для індивідуального значення регресанда:
HM : 62.0545
BM : 72.9071
